In [3]:
import numpy as np
import pandas as pd
import anndata
import h5py
from tqdm import tqdm
import bioframe as bf
import os, sys
from scipy.stats import mannwhitneyu, kruskal, wilcoxon
from plotnine import *
from grelu.sequence.utils import resize
from danrerlib import mapping

sys.path.append('/code/decima/src/decima')
sys.path.append('/hpc/mydata/mathias.voges/Projects/research/seq2fun/step/decima-main/src/decima/')

#from resources import load_gtf
from genome import read_gtf

%matplotlib inline

## Paths

In [4]:
save_dir="/hpc/mydata/mathias.voges/Projects/research/seq2fun/step/data"
matrix_file = os.path.join(save_dir, "data_out_decima_3q5pxnzb_v20250329_pretrained-human_rep0.h5ad")
h5_file = os.path.join(save_dir, "/hpc/mydata/mathias.voges/Projects/research/seq2fun/step/data/3q5pxnzb-attr-t05.h5")

## Load test genes

In [5]:
ad = anndata.read_h5ad(matrix_file)
ad = ad[:, ad.var.dataset == "test"].copy()

In [6]:
genes = ad.var.reset_index()
genes['gene'] = ad.var_names
genes['st'] = genes.gene_start - genes.start
genes['en'] = [min(524287, x) for x in genes.gene_end - genes.start]

In [7]:
genes.columns

Index(['level_0', 'index', 'chrom', 'start', 'end', 'strand', 'gene_type',
       'frac_nan', 'mean_counts', 'n_tracks', 'gene_start', 'gene_end',
       'gene_length', 'gene_mask_start', 'gene_mask_end', 'frac_N',
       'Upstream bases', 'Downstream bases', 'gene', 'fold', 'dataset',
       'pearson', 'size_factor_pearson', 'st', 'en'],
      dtype='object')

## Create list of zebrafish-human orthologs with percentage identity between genes. 

In [39]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def get_highest_identity_human_ortholog(gene_id):
    url = f"https://rest.ensembl.org/homology/symbol/danio_rerio/{gene_id}?target_species=homo_sapiens"
    headers = {"Content-Type": "application/json"}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.ok:
            data = response.json()
            homologies = data['data'][0].get('homologies', [])
            human_homologies = [h for h in homologies if h['target']['species'] == 'homo_sapiens']
            if human_homologies:
                # Select the homology with the highest source perc_id
                best_homology = max(human_homologies, key=lambda h: h['source']['perc_id'])
                return {
                    'gene': gene_id,
                    'human_ortholog_id': best_homology['target']['id'],
                    'source_perc_id': best_homology['source']['perc_id'],
                    'source_perc_pos': best_homology['source']['perc_pos']
                }
    except Exception as e:
        print(f"Error processing gene {gene_id}: {e}")
    return {
        'gene': gene_id,
        'human_ortholog_id': None,
        'source_perc_id': None,
        'source_perc_pos': None
    }

# Replace this list with your actual list of zebrafish gene symbols
gene_list = genes['level_0'].values

results = []

# Define the number of threads; adjust based on your system's capabilities
max_workers = 10

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    # Submit all tasks to the executor
    future_to_gene = {executor.submit(get_highest_identity_human_ortholog, gene): gene for gene in gene_list}
    for future in tqdm(as_completed(future_to_gene), total=len(future_to_gene)):
        result = future.result()
        results.append(result)

# Create a DataFrame and save to CSV
df = pd.DataFrame(results)
df.to_csv('zebrafish_human_orthologs.csv', index=False)
print("Results saved to 'zebrafish_human_orthologs.csv'")


  2%|▏         | 182/7289 [00:31<23:45,  4.98it/s]

Error processing gene CABZ01020834.1: HTTPSConnectionPool(host='rest.ensembl.org', port=443): Read timed out. (read timeout=10)


 10%|█         | 740/7289 [01:36<10:17, 10.60it/s]

Error processing gene si:ch73-289h5.5: list index out of range


 73%|███████▎  | 5292/7289 [08:20<02:47, 11.89it/s]

Error processing gene AL844185.1: list index out of range


100%|██████████| 7289/7289 [11:01<00:00, 11.03it/s]

Results saved to 'zebrafish_human_orthologs.csv'


In [40]:
df

,gene,human_ortholog_id,source_perc_id,source_perc_pos
0,eps8a,ENSG00000151491,60.0746,74.6269
1,ptpn12,None,NaN,NaN
2,gpr19,ENSG00000183150,63.1961,76.9976
3,ptpro-1,None,NaN,NaN
4,RERG,ENSG00000134533,77.7174,88.5870
...,...,...,...,...
7284,pitx1,None,NaN,NaN
7285,faxdc2,ENSG00000170271,61.1621,77.3700
7286,slc22a21,ENSG00000168065,28.2051,44.6886
7287,slc22a4,ENSG00000197208,48.4629,66.9078
